# Homework 2 — Orientation in 3D

### Purpose

One of the goals of this course is to:

> **Put the robot arm's end-effector at a commanded position *and* orientation.**

Position is three numbers. Orientation is not. Before you can command a hand pose,
or measure how far the hand is from the pose you want, you need to answer four
questions:

| | question | where it is used |
|---|---|---|
| **1** | How do I *write down* a 3D orientation? | Q1–Q5 |
| **2** | How do I *apply* one to a vector, and compose two of them? | Q5–Q6, Q9 |
| **3** | How do I measure the *error* between two orientations? | Q7 |
| **4** | How does an orientation *change* when the robot moves? | Q8 |

**Note:** an orientation error is not a subtraction. $R_\text{desired} - R_\text{current}$
is not a rotation, and its entries are not an angle.

**Grading:** 100 points + 10 bonus.
$\color{red}{\text{Use only numpy, matplotlib and scipy. No other packages.}}$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401  (registers the 3d projection)
from IPython.display import HTML
import scipy.linalg

np.set_printoptions(precision=5, suppress=True)

### Plotting helpers (given — read them, you do not need to modify them)
A coordinate frame is drawn as its three basis vectors. The columns of a rotation
matrix $R$ **are** the local frame's axes written in global coordinates, which is convenient: to draw a frame, draw the columns of $R$.

In [ ]:
def draw_vector(ax, origin, end, **kw):
    origin = np.asarray(origin, float); end = np.asarray(end, float)
    ax.quiver(*origin, *(end - origin), **kw)

def draw_axes(ax=None, R=np.eye(3), offset=(0,0,0), draw_global_frame=True, scale=1.0, **kw):
    '''Draw the frame whose orientation is R, centred at `offset`.
    The columns of R are the local x, y, z axes expressed in global coordinates.'''
    offset = np.asarray(offset, float)
    if ax is None:
        ax = plt.figure(figsize=(7,7)).add_subplot(projection='3d')
    if draw_global_frame:
        for i, c in enumerate('rgb'):
            e = np.zeros(3); e[i] = scale
            draw_vector(ax, offset, offset + e, lw=6, color=c, alpha=0.18,
                        label=f'{"xyz"[i]}0 global')
    for i, c in enumerate('rgb'):
        draw_vector(ax, offset, offset + scale*R[:, i], lw=6, color=c,
                    label=f'{"xyz"[i]} local')
    ax.set_xlim([-1.6,1.6]); ax.set_ylim([-1.6,1.6]); ax.set_zlim([-1.6,1.6])
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
    return ax

def check(name, got, want, tol=1e-6):
    ok = np.allclose(np.asarray(got, float), np.asarray(want, float), atol=tol)
    print(f'{"PASS" if ok else "FAIL"}  {name}')
    if not ok:
        print('   your value:', np.asarray(got)); print('   expected  :', np.asarray(want))
    return ok

---
# Q1 [10 pts] ZYX Euler angles $\rightarrow$ $SO(3)$

A common way to *type in* an orientation is Euler angles. With the
ZYX convention, you rotate about $z$ by $\theta_z$, then about the **new** $y$ by
$\theta_y$, then about the **new** $x$ by $\theta_x$:

$$R = R_z(\theta_z)\,R_y(\theta_y)\,R_x(\theta_x)$$

**(a)** Complete `ZYX_to_SO3`.

**(b)** Use it to draw the three frames below.

$$R_a:\ ZYX = (0.3,\,0.2,\,0.7),\qquad R_b:\ ZYX=(0.7,\,0,\,\pi/2),\qquad R_c:\ ZYX=(\pi/3,\,0,\,0)\ \ [\text{rad}]$$

In [ ]:
def ZYX_to_SO3(th_z, th_y, th_x):
    '''Return the 3x3 rotation matrix for ZYX Euler angles (radians).'''
    # TODO Q1: build Rz, Ry, Rx and return their product (mind the order).
    raise NotImplementedError("Q1: ZYX_to_SO3")


In [ ]:
# --- self test ---
zyx_test = np.array([[0.3       , 0.2       , 0.7       ],
           [0.7       , 3.14159265, 1.57079633],
           [1.04719755, 0.        , 0.        ]])
soln = np.array([[[ 9.36293364e-01, -1.03756342e-01,  3.35543378e-01],
            [ 2.89629478e-01,  7.68504187e-01, -5.70540166e-01],
            [-1.98669331e-01,  6.31376224e-01,  7.49596265e-01]],
          
           [[-7.64842187e-01,  5.42191972e-17,  6.44217687e-01],
            [-6.44217687e-01,  1.25726990e-16, -7.64842187e-01],
            [-1.22464680e-16, -1.00000000e+00, -6.12323400e-17]],
          
           [[ 5.00000000e-01, -8.66025404e-01,  0.00000000e+00],
            [ 8.66025404e-01,  5.00000000e-01,  0.00000000e+00],
            [ 0.00000000e+00,  0.00000000e+00,  1.00000000e+00]]])
check('Q1 ZYX_to_SO3', [ZYX_to_SO3(*t) for t in zyx_test], soln)

In [ ]:
# (b) draw the three frames
for label, zyx in [('(a)', [0.3, 0.2, 0.7]), ('(b)', [0.7, 0, np.pi/2]), ('(c)', [np.pi/3, 0, 0])]:
    fig = plt.figure(figsize=(6,6)); ax = fig.add_subplot(projection='3d')
    draw_axes(ax, R=ZYX_to_SO3(*zyx))
    ax.set_title(f'{label} ZYX = {zyx}'); ax.view_init(25, 40); ax.legend(fontsize=7)
    plt.show()

---
# Q2 [10 pts] Why we do *not* control robots with Euler angles

Euler angles have two defects that matter here: **ambiguity** — different triples
can name the same orientation, and at some configurations a degree of freedom
disappears altogether — and a **non-geodesic metric** — distance measured in angle
coordinates is not distance between orientations. This question makes both
concrete.

**(a)** Evaluate `ZYX_to_SO3` at these three *different* angle triples and print
all three matrices:

$$(0.5,\ \pi/2,\ 0.2),\qquad (0.8,\ \pi/2,\ 0.5),\qquad (1.1,\ \pi/2,\ 0.8)$$

Report what you observe, then work out algebraically why it happens: set
$\theta_y = \pi/2$ in $R_zR_yR_x$ and use the angle-sum identities. This is
**gimbal lock** — at $\theta_y = \pi/2$ a degree of freedom is gone and only
$\theta_z - \theta_x$ has any effect. Give two more triples that collapse to the
same matrix, and state the rule that generates them.

**(b)** (the metric problem) Take two orientations that are numerically close in
Euler coordinates and ask how far apart they actually are. Compute the rotation
angle between $ZYX=(0,\ \pi/2 - 0.01,\ 0)$ and $ZYX=(0,\ \pi/2+0.01,\ 0)$,
and between $ZYX=(3.0,\ \pi/2-0.01,\ 0)$ and $ZYX=(0,\ \pi/2+0.01,\ 3.0)$, using

$$\theta = \arccos\!\Big(\tfrac{\operatorname{tr}(R_1^\top R_2)-1}{2}\Big)$$

The second pair differs by $3$ rad in $\theta_z$ *and* $3$ rad in $\theta_x$ — a
Euler-space distance of $4.24$ rad — while the rotation between the two frames is
about $0.28$ rad. Write 2–3 sentences on why this makes Euler angles unsuitable
for a control error or a cost function.

In [ ]:
# (a) gimbal lock
# === HW SOLUTION START: q2.gimbal ===
triples = [(0.5, np.pi/2, 0.2), (0.8, np.pi/2, 0.5), (1.1, np.pi/2, 0.8)]
for zyx in triples:
    print(f'ZYX={zyx}  ->\n{ZYX_to_SO3(*zyx)}\n')
print('All three are the SAME matrix: at th_y = pi/2 only (th_z - th_x) matters,')
print('and th_z - th_x = 0.3 for all three.\n')

# two more triples in the same family: any pair with th_z - th_x = 0.3
extra = [(1.4, np.pi/2, 1.1), (-0.7, np.pi/2, -1.0)]
R_ref = ZYX_to_SO3(*triples[0])
for zyx in extra:
    print(f'ZYX={zyx}  ->  same matrix?  {np.allclose(ZYX_to_SO3(*zyx), R_ref)}')

# closed form: R = Rz(th_z - th_x) Ry(pi/2), a function of the difference alone
d = 0.3
R_closed = np.array([[ 0, -np.sin(d), np.cos(d)],
                     [ 0,  np.cos(d), np.sin(d)],
                     [-1,          0,         0]])
check('Q2(a) closed form  Rz(th_z - th_x) Ry(pi/2)', R_ref, R_closed)
# === HW SOLUTION END: q2.gimbal ===

In [ ]:
# (b) Euler distance vs true rotation angle
def rotation_angle_between(R1, R2):
    '''True geodesic angle between two orientations, in radians.'''
    # TODO Q2(b): theta = arccos((trace(R1^T R2) - 1)/2). Clip the argument to [-1,1].
    raise NotImplementedError("Q2(b): rotation_angle_between")

pairs = [((0, np.pi/2 - 0.01, 0), (0, np.pi/2 + 0.01, 0)),
         ((3.0, np.pi/2 - 0.01, 0), (0, np.pi/2 + 0.01, 3.0))]
for z1, z2 in pairs:
    euler_dist = np.linalg.norm(np.array(z1) - np.array(z2))
    true_ang = rotation_angle_between(ZYX_to_SO3(*z1), ZYX_to_SO3(*z2))
    print(f'Euler-space distance {euler_dist:6.3f} rad   ->   true rotation angle {true_ang:6.4f} rad')

**(Your answer for Q2)**

*write here*

---
# Q3 [15 pts] $so(3) \leftrightarrow SO(3)$: the exponential and log maps

$SO(3)$ is a 3x3 matrix — 9 numbers with 6 constraints, so 3 DoF. Its Lie algebra
$so(3)$ is the 3-number description: a **unit axis** $\hat\omega$ and an
**angle** $\theta$.

**Exponential map (Rodrigues' formula)** — axis/angle $\rightarrow$ matrix:
$$R = e^{[\hat\omega]\theta} = I + \sin\theta\,[\hat\omega] + (1-\cos\theta)[\hat\omega]^2$$

**Log map** — matrix $\rightarrow$ axis/angle:
$$\theta = \arccos\!\Big(\frac{\operatorname{tr}(R)-1}{2}\Big),
\qquad [\hat\omega] = \frac{1}{2\sin\theta}\big(R - R^\top\big)$$

**Two singular cases.**

* $R = I$: $\theta=0$ and the axis is undefined — return $\theta=0$ and any axis.

* $\theta = \pi$: the log formula is $0/0$. The denominator $\sin\pi = 0$, and the
  numerator vanishes as well, because Rodrigues at $\theta = \pi$ gives
  $$R = I + 2[\hat\omega]^2 = 2\hat\omega\hat\omega^\top - I$$
  (the second form follows from $[\hat\omega]^2 = \hat\omega\hat\omega^\top - I$
  for a unit axis), which is symmetric, so $R - R^\top = 0$. The antisymmetric part
  carries no information here and the axis has to come from the symmetric part.

  Use $R + I = 2\hat\omega\hat\omega^\top$. Column $i$ of $\hat\omega\hat\omega^\top$
  is $\hat\omega\,\hat\omega_i$, so column $i$ of $R + I$ is
  $2\hat\omega_i\,\hat\omega$, with norm $2|\hat\omega_i|$: **every column of
  $R+I$ is parallel to the axis**, and the columns differ only in length and sign.
  Normalise any non-zero column and you have the axis. Take the **longest** one:
  a column with $\hat\omega_i \approx 0$ is a near-zero vector whose direction is
  set by round-off, whereas the longest column always has
  $|\hat\omega_i| \ge 1/\sqrt{3}$.

  Some textbooks give a fixed single-column version,
  $\hat\omega = [\,r_{13},\ r_{23},\ 1+r_{33}\,]^\top / \sqrt{2(1+r_{33})}$, which
  is the third column of $R+I$ normalised — the square root is the length of that
  column. Check what it gives for a $\pi$-rotation about $x$.

  Note also that $\hat\omega$ and $-\hat\omega$ describe the same rotation when
  $\theta = \pi$, so the axis is determined only up to sign.

**(a)** Complete `hat`, `vee`, `so3_to_SO3`, `SO3_to_so3`.

**(b)** Rotate about $x$ by $30^\circ$, then about the **global** $y$ by
$-20^\circ$. Confirm that $\operatorname{tr}(R)=2.6195$, $\theta=0.6271$ and
$\hat\omega=[0.8264,\,-0.5438,\,0.1457]$.

In [ ]:
def hat(a):
    '''3-vector -> 3x3 skew-symmetric matrix (so(3) matrix form).'''
    # TODO Q3: return the 3x3 skew-symmetric matrix built from a.
    raise NotImplementedError("Q3: hat")

def vee(S):
    '''3x3 skew-symmetric matrix -> 3-vector. Inverse of hat().'''
    # TODO Q3: pull the 3 independent entries back out of the skew matrix.
    raise NotImplementedError("Q3: vee")

def so3_to_SO3(omega, theta):
    '''Rodrigues' formula: unit axis + angle -> rotation matrix.'''
    # TODO Q3: Rodrigues.  R = I + sin(th)[w] + (1-cos(th))[w]^2
    #          Handle th ~ 0 separately.
    raise NotImplementedError("Q3: so3_to_SO3")

def SO3_to_so3(R):
    '''Log map: rotation matrix -> (unit axis, angle). Handle R=I and theta=pi.'''
    # TODO Q3: log map. theta = arccos((tr(R)-1)/2), [w] = (R - R^T)/(2 sin theta).
    #          You MUST handle two singular cases: R = I, and theta = pi.
    raise NotImplementedError("Q3: SO3_to_so3")


In [ ]:
# --- self test: hat/vee round trip, exp/log round trip, and the singular cases ---
v = np.array([0.3, -0.7, 1.1])
check('Q3 vee(hat(v)) == v', vee(hat(v)), v)
check('Q3 hat is skew', hat(v) + hat(v).T, np.zeros((3,3)))

rng = np.random.default_rng(0); ok = True
for _ in range(200):
    ax_ = rng.normal(size=3); ax_ /= np.linalg.norm(ax_)
    th_ = rng.uniform(0.05, np.pi - 0.05)
    R_ = so3_to_SO3(ax_, th_)
    ok &= np.allclose(R_.T @ R_, np.eye(3), atol=1e-9) and abs(np.linalg.det(R_)-1) < 1e-9
    w2, t2 = SO3_to_so3(R_)
    ok &= np.allclose(so3_to_SO3(w2, t2), R_, atol=1e-8)
print(f'{"PASS" if ok else "FAIL"}  Q3 exp/log round trip over 200 random rotations')

w0, t0 = SO3_to_so3(np.eye(3));  check('Q3 log(I) angle = 0', t0, 0.0)
ok_pi = True
for ax_pi in [[1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,1,1], [0,1,-1]]:
    ax_pi = np.array(ax_pi, float); ax_pi /= np.linalg.norm(ax_pi)
    Rpi = so3_to_SO3(ax_pi, np.pi)
    wpi, tpi = SO3_to_so3(Rpi)
    ok_pi &= np.isfinite(wpi).all() and np.allclose(so3_to_SO3(wpi, tpi), Rpi, atol=1e-7)
print(f'{"PASS" if ok_pi else "FAIL"}  Q3 theta = pi about 6 different axes')

In [ ]:
# (b) Rotate about x by 30 degrees, then about the global y by -20 degrees.
# TODO Q3(b): build R (about x by 30 deg, then about the GLOBAL y by -20 deg)
#             and report trace(R), theta and omega from your log map.
raise NotImplementedError("Q3(b): verify the exp/log maps")

check('Q3(b)', np.concatenate([w_q3b, [th_q3b]]),
      np.array([ 0.82643617, -0.54384562,  0.145723  ,  0.62705701]), tol=1e-4)

---
# Q4 [10 pts] ZYX $\rightarrow$ quaternion

A quaternion packs the same axis/angle information into 4 numbers, with no
trigonometric singularities and a single norm constraint that is cheap to
renormalise. That is why most dynamics simulators store orientation this way
(as does the IMU on DARoS Lab's humanoid robots):

$$q = \big[\cos(\theta/2),\ \ \hat\omega\sin(\theta/2)\big] \in \mathbb{R}^4$$

Complete `ZYX_to_Quaternion` **by reusing `SO3_to_so3` from Q3** — do not derive a
separate formula. Note that $q$ and $-q$ are the *same* rotation (the double cover).

In [ ]:
def ZYX_to_Quaternion(th_z, th_y, th_x):
    '''ZYX Euler angles -> quaternion [w, x, y, z].'''
    # TODO Q4: reuse SO3_to_so3, then q = [cos(th/2), w*sin(th/2)].
    raise NotImplementedError("Q4: ZYX_to_Quaternion")


In [ ]:
soln_q = np.array([[ 0.92929998,  0.32333918,  0.14371374,  0.10582853],
           [ 0.24246536, -0.24246536,  0.66423682, -0.66423682],
           [ 0.8660254 ,  0.        ,  0.        ,  0.5       ]])
check('Q4 ZYX_to_Quaternion', [ZYX_to_Quaternion(*t) for t in zyx_test], soln_q)
print('unit norm?', np.allclose([np.linalg.norm(ZYX_to_Quaternion(*t)) for t in zyx_test], 1.0))

---
# Q5 [10 pts] Composing rotations with quaternions (Hamilton product)

Composing two rotations is a matrix product in $SO(3)$; for quaternions it is the
**Hamilton product**. For $q_1=(w_1,\mathbf{v}_1)$ and $q_2=(w_2,\mathbf{v}_2)$:

$$q_1 \otimes q_2 = \big(\,w_1w_2 - \mathbf{v}_1\!\cdot\!\mathbf{v}_2,\ \ \
w_1\mathbf{v}_2 + w_2\mathbf{v}_1 + \mathbf{v}_1\times\mathbf{v}_2\,\big)$$

**(a)** Complete `quat_mul` and `quat_to_SO3`.

**(b)** Let $q_x$ be the rotation about $x$ by $30^\circ$ and $q_y$ the rotation
about $y$ by $-20^\circ$. Confirm that

$$q_y \otimes q_x = 0.95125 + 0.25489\,i - 0.16773\,j + 0.044943\,k$$

and that it equals the quaternion of the matrix $R_yR_x$ you built in Q3(b).
Confirm that **quaternion multiplication order matches matrix multiplication
order**.

In [ ]:
def quat_mul(q1, q2):
    '''Hamilton product. Same composition order as matrix multiplication.'''
    # TODO Q5: Hamilton product of q1 and q2.
    raise NotImplementedError("Q5: quat_mul")

def quat_to_SO3(q):
    '''Quaternion [w,x,y,z] -> 3x3 rotation matrix.'''
    # TODO Q5: build the rotation matrix from a unit quaternion.
    raise NotImplementedError("Q5: quat_to_SO3")


In [ ]:
q_x = ZYX_to_Quaternion(0, 0, np.deg2rad(30))
q_y = ZYX_to_Quaternion(0, np.deg2rad(-20), 0)
q_yx = quat_mul(q_y, q_x)
print('q_y (x) q_x =', q_yx)
check('Q5 Hamilton product', q_yx,
      np.array([ 0.95125124,  0.254887  , -0.16773126,  0.04494346]), tol=1e-4)
check('Q5 quaternion product == matrix product', quat_to_SO3(q_yx), R_q3b, tol=1e-8)

---
# Q6 [10 pts] Rotating a vector: local vs global coordinates

Take $p = (1,\ 0.4,\ 0.7)$ and the three matrices $R_a, R_b, R_c$ from Q1.

**(a)** Compute and plot $p' = R p$ for each.

**(b)** For each case report the coordinates of the rotated point **in the global
frame** and **in the local (rotated) frame**.

**(c)** The numbers read in the local frame do not change. Explain why in 2–3
sentences.

Which frame an error is expressed in is a choice you make in every controller you
write, and getting it wrong is a common source of sign errors. Q7 depends on this
part being clear.

In [ ]:
p = np.array([1.0, 0.4, 0.7])
R_a = ZYX_to_SO3(0.3, 0.2, 0.7)
R_b = ZYX_to_SO3(0.7, 0.0, np.pi/2)
R_c = ZYX_to_SO3(np.pi/3, 0.0, 0.0)

for name, R in [('a', R_a), ('b', R_b), ('c', R_c)]:
    # === HW SOLUTION START: q6.rotate ===
    p_global = R @ p           # the rotated point, in global coordinates
    p_local  = R.T @ p_global  # the same point, read in the rotated frame
    # === HW SOLUTION END: q6.rotate ===
    print(f'({name})  p_global = {p_global}   p_local = {p_local}')

    fig = plt.figure(figsize=(6,6)); ax = fig.add_subplot(projection='3d')
    draw_axes(ax, R=R)
    draw_vector(ax, [0,0,0], p_global, color='c', lw=6, label="p' rotated")
    ax.set_title(f'({name})'); ax.view_init(25, 40); ax.legend(fontsize=7)
    plt.show()

**(Your answer for Q6c)**

*write here*

---
# Q7 [15 pts] Orientation error

The arm's hand is at orientation $R_\text{cur}$; you want it at $R_\text{des}$.
**What is the error between the target and the current orientation?**

$R_\text{des} - R_\text{cur}$ is not usable — the difference of two rotation
matrices is not a rotation, and its entries are not an angle. The question to ask
instead is: *which single rotation takes the current orientation to the desired
one?* There are two answers, and which one you want depends on the frame you act
in:

$$\underbrace{R_\text{err}^{\text{body}} = R_\text{cur}^{\top} R_\text{des}}_{\text{implicit — rotate about the \textit{local} axes}}
\qquad\qquad
\underbrace{R_\text{err}^{\text{world}} = R_\text{des} R_\text{cur}^{\top}}_{\text{explicit — rotate about the \textit{global} axes}}$$

The log of either one is a 3-vector $e = \hat\omega\theta$: an axis scaled by an
angle. That is a quantity a controller can use.

**(a)** Complete `orientation_error_body` and `orientation_error_world`, each
returning the 3-vector $\hat\omega\theta$.

**(b)** With $R_\text{cur} = $ `ZYX_to_SO3(0.2, -0.3, 0.5)` and
$R_\text{des} = $ `ZYX_to_SO3(-0.4, 0.6, 0.1)`, print both errors. They have
**different directions but identical magnitude**. Explain why, using
$\operatorname{tr}(AB) = \operatorname{tr}(BA)$.

**(c)** Show numerically that $e_\text{world} = R_\text{cur}\,e_\text{body}$, and
state in one sentence what that means physically.

**(d)** The proportional orientation controller is
$\omega_\text{cmd} = k_p\, e_\text{world}$, integrated **explicitly** (global
axes): $R \leftarrow e^{[\omega_\text{cmd}]\Delta t} R$. Run 200 steps with
$k_p = 2.0$ and $\Delta t = 0.01$, and plot $\|e\|$ against time. The result
should be exponential decay, $\|e(t)\| \approx \|e(0)\|e^{-k_p t}$, leaving about
$2\%$ of the initial error after 2 s.

In [ ]:
def orientation_error_body(R_cur, R_des):
    '''Error as an axis-angle 3-vector, expressed in the CURRENT (body) frame.'''
    # TODO Q7: implicit error. Which product gives the rotation expressed in the
    #          CURRENT frame? Take its log and return omega*theta.
    raise NotImplementedError("Q7: orientation_error_body")

def orientation_error_world(R_cur, R_des):
    '''Error as an axis-angle 3-vector, expressed in the GLOBAL frame.'''
    # TODO Q7: explicit error, expressed in the GLOBAL frame.
    raise NotImplementedError("Q7: orientation_error_world")


In [ ]:
R_cur = ZYX_to_SO3(0.2, -0.3, 0.5)
R_des = ZYX_to_SO3(-0.4, 0.6, 0.1)
e_body  = orientation_error_body(R_cur, R_des)
e_world = orientation_error_world(R_cur, R_des)
print(f'e_body  = {e_body}   |e| = {np.linalg.norm(e_body):.6f}')
print(f'e_world = {e_world}   |e| = {np.linalg.norm(e_world):.6f}')
check('Q7 e_body',  e_body,  np.array([-0.26864093,  0.65387482, -0.84611213]))
check('Q7 e_world', e_world, np.array([-0.32185352,  0.9341554 , -0.48927495]))
check('Q7 same magnitude', np.linalg.norm(e_body), np.linalg.norm(e_world))
check('Q7 e_world == R_cur @ e_body', R_cur @ e_body, e_world)

In [ ]:
# (d) proportional orientation controller
kp, dt, N = 2.0, 0.01, 200
R = R_cur.copy()
err_hist = []
for k in range(N):
    # TODO Q7(d): compute e, store |e|, set omega_cmd = kp*e, then integrate R.
    #             Explicit integration -- which side does the exponential go on?
    raise NotImplementedError("Q7(d): orientation controller")

plt.figure(figsize=(6,4))
plt.plot(np.arange(N)*dt, err_hist)
plt.xlabel('time [s]'); plt.ylabel(r'$\|e\|$  [rad]')
plt.title('orientation error under $\\omega = k_p e$'); plt.grid(alpha=0.3); plt.show()
print(f'final error = {err_hist[-1]:.3e} rad  (started at {err_hist[0]:.4f})')

**(Your answer for Q7b and Q7c)**

*write here*

---
# Q8 [10 pts] Integrating angular velocity: implicit vs explicit

A gyro gives you $\omega$; you need $R(t)$. There are two ways to apply the
increment, and they are **not** the same:

$$\text{implicit (local axes): } R \leftarrow R\,e^{[\omega]\Delta t}
\qquad\qquad
\text{explicit (global axes): } R \leftarrow e^{[\omega]\Delta t}\,R$$

**(a)** Integrate $\omega = [0,\ 10,\ 0]$ rad/s for $0.45$ s — 30 steps of
$\Delta t = 0.015$ s — from
$R_0 = \begin{bmatrix}0&1&0\\-1&0&0\\0&0&1\end{bmatrix}$, both ways.

**(b)** Animate both and explain the difference in 2–3 sentences.

In [ ]:
omega = np.array([0.0, 10.0, 0.0])
N_t, delta_t = 31, 0.015          # 30 steps of 0.015 s  ->  0.45 s total
R0 = np.array([[0.,1.,0.],[-1.,0.,0.],[0.,0.,1.]])

R_implicit = np.zeros((N_t,3,3)); R_explicit = np.zeros((N_t,3,3))
R_implicit[0] = R0; R_explicit[0] = R0
exp_w = scipy.linalg.expm(hat(omega*delta_t))

for i in range(N_t-1):
    # TODO Q8: one of these is pre-multiply and one is post-multiply. Which?
    raise NotImplementedError("Q8: angular velocity integration")

T_end = (N_t - 1) * delta_t
print(f'after {T_end:.2f} s  ({T_end*omega[1]:.2f} rad of rotation)')
print('implicit:\n', R_implicit[-1])
print('explicit:\n', R_explicit[-1])

In [ ]:
# No need to change
def animate_frames(T_hist, title='', **kw):
    '''T_hist: (N, n_frames, 4, 4) SE(3), or (N, 3, 3) SO(3) at the origin.'''
    T_hist = np.asarray(T_hist)
    if T_hist.ndim == 3:                       # (N,3,3) -> (N,1,4,4)
        H = np.zeros((T_hist.shape[0], 1, 4, 4)); H[:,0] = np.eye(4)
        H[:,0,:3,:3] = T_hist; T_hist = H
    N, nf = T_hist.shape[:2]
    fig = plt.figure(figsize=(6,6)); ax = fig.add_subplot(projection='3d')
    for i, c in enumerate('rgb'):
        e = np.zeros(3); e[i] = 1
        draw_vector(ax, [0,0,0], e, lw=5, color=c, alpha=0.15)
    lines = [[ax.plot([0,1],[0,0],[0,0], lw=5, color=c)[0] for c in 'rgb'] for _ in range(nf)]
    ax.set_xlim([-1.6,1.6]); ax.set_ylim([-1.6,1.6]); ax.set_zlim([-1.6,1.6])
    ax.view_init(25,40); ax.set_title(title)
    def frame(k):
        for i in range(nf):
            T = T_hist[k,i]; R, o = T[:3,:3], T[:3,3]
            for j in range(3):
                e = o + R[:,j]
                lines[i][j].set_data([o[0],e[0]], [o[1],e[1]])
                lines[i][j].set_3d_properties([o[2],e[2]])
        return [l for grp in lines for l in grp]
    anim = animation.FuncAnimation(fig, frame, frames=N, interval=60, blit=False)
    plt.close(fig)
    return anim

HTML(animate_frames(R_implicit, 'IMPLICIT: R <- R exp([w]dt)  (local axis)').to_jshtml())

In [ ]:
HTML(animate_frames(R_explicit, 'EXPLICIT: R <- exp([w]dt) R  (global axis)').to_jshtml())

**(Your answer for Q8b)**

*write here*

---
# Q9 [10 pts] $SE(3)$: orientation + position, and a chain of frames

An arm is a chain of frames, each described relative to the previous one by a
homogeneous transform $T=\begin{bmatrix}R & p\\ 0 & 1\end{bmatrix}\in SE(3)$.
Composing along the chain is a matrix product, and that product is forward
kinematics.

**(a)** Complete `POS_ZYX_to_SE3`.

**(b)** Build $T^0_i$ for $i=1,2,3$ from these relative transforms and plot all
four frames together, with $\theta_1=\theta_2=\theta_3=0$:

* frame $\{0\}$: position $(0,0,0)$, $ZYX=(0,0,0)$
* $\{0\}\rightarrow\{1\}$: position $(0.4,\,0,\,0.4)$, $ZYX=(0,\theta_1,0)$
* $\{1\}\rightarrow\{2\}$: position $(0.7,\,0,\,0.8)$, $ZYX=(0,\theta_2,0)$
* $\{2\}\rightarrow\{3\}$: position $(1.0,\,0,\,0.4)$, $ZYX=(0,\theta_3,0)$

**(c)** Animate $t\in[0,1.5]$ s with $\theta_1=\sin(\omega t)$,
$\theta_2=\sin(\omega t+0.3)$, $\theta_3=0$, $\omega=5$ rad/s.

In [ ]:
def POS_ZYX_to_SE3(x, y, z, th_z, th_y, th_x):
    '''Position + ZYX Euler angles -> 4x4 homogeneous transform.'''
    # TODO Q9: pack R and p into a 4x4 homogeneous transform.
    raise NotImplementedError("Q9: POS_ZYX_to_SE3")

def chain(th1, th2, th3):
    '''Return [T0, T0_1, T0_2, T0_3] for the joint angles given.'''
    # TODO Q9: build each relative transform, then compose along the chain.
    raise NotImplementedError("Q9: chain")


In [ ]:
se3_test = np.array([[0.3       , 0.2       , 0.7       , 0.5       , 0.2       , 1.2       ],
           [0.5       , 0.6       , 1.8       , 0.7       , 3.14159265, 1.57079633],
           [0.1       , 0.        , 2.        , 1.04719755, 0.        , 0.        ]])
soln_se3 = np.array([[[ 8.60089338e-01, -1.12237211e-02,  5.10019959e-01,  3.00000000e-01],
            [ 4.69868947e-01,  4.06772914e-01, -7.83427705e-01,  2.00000000e-01],
            [-1.98669331e-01,  9.13460357e-01,  3.55134724e-01,  7.00000000e-01],
            [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  1.00000000e+00]],
          
           [[-7.64842187e-01,  5.42191972e-17,  6.44217687e-01,  5.00000000e-01],
            [-6.44217687e-01,  1.25726990e-16, -7.64842187e-01,  6.00000000e-01],
            [-1.22464680e-16, -1.00000000e+00, -6.12323400e-17,  1.80000000e+00],
            [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  1.00000000e+00]],
          
           [[ 5.00000000e-01, -8.66025404e-01,  0.00000000e+00,  1.00000000e-01],
            [ 8.66025404e-01,  5.00000000e-01,  0.00000000e+00,  0.00000000e+00],
            [ 0.00000000e+00,  0.00000000e+00,  1.00000000e+00,  2.00000000e+00],
            [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  1.00000000e+00]]])
check('Q9 POS_ZYX_to_SE3', [POS_ZYX_to_SE3(*t) for t in se3_test], soln_se3)

In [ ]:
# (b) plot the chain at zero angles
Ts = chain(0., 0., 0.)
fig = plt.figure(figsize=(7,7)); ax = fig.add_subplot(projection='3d')
for T in Ts:
    draw_axes(ax, R=T[:3,:3], offset=T[:3,3], draw_global_frame=False, scale=0.35)
ax.set_xlim([-0.5,2.5]); ax.set_ylim([-1.5,1.5]); ax.set_zlim([-0.5,2.0])
ax.view_init(25, 40); ax.set_title('frames {0},{1},{2},{3}'); plt.show()

In [ ]:
# (c) animate
N, w = 50, 5.0
t = np.linspace(0, 1.5, N)
T_hist = np.array([chain(np.sin(w*ti), np.sin(w*ti + 0.3), 0.0) for ti in t])
HTML(animate_frames(T_hist, 'SE(3) chain').to_jshtml())

---
# Q10 [BONUS 10 pts] The full 6-D pose error

The pieces above combine into the quantity that inverse kinematics and
operational-space control minimise. The tool frame is at
$(p_\text{cur}, R_\text{cur})$ and you want it at $(p_\text{des}, R_\text{des})$.
The **6-D pose error** stacks orientation and position:

$$e = \begin{bmatrix} e_\omega \\ e_v \end{bmatrix}
    = \begin{bmatrix} \log\!\big(R_\text{des}R_\text{cur}^{\top}\big)^{\vee} \\
                      p_\text{des} - p_\text{cur} \end{bmatrix} \in \mathbb{R}^6$$

**Ordering: angular part first, then linear.** That is the twist convention
$\mathcal{V}=[\omega; v]$ used throughout this course. A Jacobian whose rows are
written in this order has to be paired with an error in the same order, or the two
do not line up.

Only the position part is a subtraction; the orientation part goes through the log
map from Q3.

Complete `pose_error` and report $e$ and $\|e\|$ for the values below.

Then answer in 3–4 sentences: the angular part is in radians and the linear part
is in metres. If you minimise $\|e\|^2$ directly, what goes wrong, and what would
you do about it?

In [ ]:
p_cur = np.array([0.512, -0.234,  0.688])
p_des = np.array([0.600,  0.200,  0.500])
R_cur_ee = ZYX_to_SO3(0.15, -0.42, 0.33)
R_des_ee = ZYX_to_SO3(-0.20, 0.35, 0.10)

def pose_error(p_cur, R_cur, p_des, R_des):
    '''Return the 6-vector [angular error (3); linear error (3)] in world frame.'''
    # TODO Q10: stack the ANGULAR error first, then the linear error.
    #           Only the linear part is a subtraction.
    raise NotImplementedError("Q10: pose_error")

e6 = pose_error(p_cur, R_cur_ee, p_des, R_des_ee)
print('e_omega (angular) [rad]:', e6[:3], f'  |.| = {np.linalg.norm(e6[:3]):.4f} rad')
print('e_v     (linear)  [m]  :', e6[3:], f'  |.| = {np.linalg.norm(e6[3:]):.4f} m')
print('full 6-D error     :', e6)

**(Your answer for Q10)**

*write here*

---
## Submission
Run all cells top to bottom — every `check(...)` should print **PASS** — then
export to PDF and upload it to Gradescope.
Written answers are required for **Q2, Q6c, Q7b/c, Q8b** and the Q10 bonus.